In [1]:
# PASO 1 — COMPROBAR ENTORNO

import torch

print("=== CUADERNO DE APLICACIÓN ===")
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("\nLISTO PARA CONTINUAR: SÍ")
else:
    print("GPU: NO DISPONIBLE")
    print("\nLISTO PARA ENTRENAR BERT: NO")
    print("No pasa nada. No ejecutes entrenamiento todavía.")

=== CUADERNO DE APLICACIÓN ===
PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4

LISTO PARA CONTINUAR: SÍ


In [2]:
# ============================================================
# PASO 2 — CONECTAR GOOGLE DRIVE Y CREAR CARPETA PERMANENTE
# ============================================================

from google.colab import drive
from pathlib import Path

print("=== PASO 2 — GOOGLE DRIVE ===")

# Montar Drive
drive.mount("/content/drive")

# Carpeta permanente del proyecto
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Yelp_NLP_Final"
)

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Subcarpetas
MODEL_DIR = PROJECT_DIR / "bert_model"
TOKENIZER_DIR = PROJECT_DIR / "tokenizer"
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"

for folder in [
    MODEL_DIR,
    TOKENIZER_DIR,
    DATA_DIR,
    RESULTS_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("\n=== CARPETAS CREADAS ===")
print("Proyecto:", PROJECT_DIR)
print("Modelo:", MODEL_DIR)
print("Tokenizer:", TOKENIZER_DIR)
print("Datos:", DATA_DIR)
print("Resultados:", RESULTS_DIR)

print("\nPASO 2 COMPLETADO: SÍ")

=== PASO 2 — GOOGLE DRIVE ===
Mounted at /content/drive

=== CARPETAS CREADAS ===
Proyecto: /content/drive/MyDrive/Yelp_NLP_Final
Modelo: /content/drive/MyDrive/Yelp_NLP_Final/bert_model
Tokenizer: /content/drive/MyDrive/Yelp_NLP_Final/tokenizer
Datos: /content/drive/MyDrive/Yelp_NLP_Final/data
Resultados: /content/drive/MyDrive/Yelp_NLP_Final/results

PASO 2 COMPLETADO: SÍ


In [2]:
# ================================================================
# YELP — MODELO FINAL DE APLICACIÓN
# TODO EN UNA SOLA CELDA
# ================================================================
#
# Este notebook de despliegue es independiente del experimento
# académico ya auditado.
#
# NO reemplaza las métricas oficiales del informe.
#
# Objetivo:
# - reconstruir un BERT operativo
# - guardarlo permanentemente en Drive
# - ofrecer análisis individual y masivo
# ================================================================

import os
import sys
import re
import html
import json
import math
import shutil
import tempfile
import subprocess
from pathlib import Path

# ================================================================
# 0. INSTALAR DEPENDENCIAS NECESARIAS
# ================================================================

print("====================================================")
print("0. PREPARANDO ENTORNO")
print("====================================================")

packages = [
    "transformers",
    "scikit-learn",
    "openpyxl",
    "gradio"
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    *packages
])

import numpy as np
import pandas as pd
import torch
import gradio as gr

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)

from google.colab import drive, files


# ================================================================
# 1. SEMILLA
# ================================================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ================================================================
# 2. MONTAR GOOGLE DRIVE
# ================================================================

print("\n====================================================")
print("1. GOOGLE DRIVE")
print("====================================================")

drive.mount(
    "/content/drive",
    force_remount=False
)

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Yelp_NLP_Final"
)

DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "bert_model"
TOKENIZER_DIR = PROJECT_DIR / "tokenizer"
RESULTS_DIR = PROJECT_DIR / "results"

for folder in [
    PROJECT_DIR,
    DATA_DIR,
    MODEL_DIR,
    TOKENIZER_DIR,
    RESULTS_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("Proyecto:", PROJECT_DIR)
print("Datos:", DATA_DIR)
print("Modelo:", MODEL_DIR)
print("Tokenizer:", TOKENIZER_DIR)
print("Resultados:", RESULTS_DIR)


# ================================================================
# 3. LOCALIZAR AUTOMÁTICAMENTE EL DATASET
# ================================================================

print("\n====================================================")
print("2. BUSCANDO DATASET")
print("====================================================")

DATASET_NAME = "yelp_dataset.xlsx"

candidate_paths = [
    Path("/content") / DATASET_NAME,
    DATA_DIR / DATASET_NAME
]

dataset_path = None

for candidate in candidate_paths:

    if candidate.exists():

        dataset_path = candidate
        break


# Buscar también en todo MyDrive
if dataset_path is None:

    print(
        "No está en las rutas principales. "
        "Buscando en Google Drive..."
    )

    for root, dirs, filenames in os.walk(
        "/content/drive/MyDrive"
    ):

        if DATASET_NAME in filenames:

            dataset_path = (
                Path(root)
                / DATASET_NAME
            )

            break


# Si no aparece, abrir selector automáticamente
if dataset_path is None:

    print(
        "\nNo encontré yelp_dataset.xlsx."
    )

    print(
        "Se abrirá automáticamente el selector "
        "de archivos de Colab."
    )

    uploaded = files.upload()

    if DATASET_NAME not in uploaded:

        raise RuntimeError(
            "Debes seleccionar yelp_dataset.xlsx."
        )

    dataset_path = (
        Path("/content")
        / DATASET_NAME
    )


print(
    "Dataset localizado:",
    dataset_path
)


# ================================================================
# 4. HACER COPIA PERMANENTE EN DRIVE
# ================================================================

PERMANENT_DATASET = (
    DATA_DIR
    / DATASET_NAME
)

if dataset_path.resolve() != PERMANENT_DATASET.resolve():

    shutil.copy2(
        dataset_path,
        PERMANENT_DATASET
    )

dataset_path = PERMANENT_DATASET

print(
    "Dataset permanente:",
    dataset_path
)


# ================================================================
# 5. CARGAR DATASET
# ================================================================

print("\n====================================================")
print("3. CARGANDO DATASET")
print("====================================================")

df_original = pd.read_excel(
    dataset_path
)

print(
    "Filas originales:",
    len(df_original)
)

print(
    "Columnas:",
    df_original.columns.tolist()
)


required_columns = [
    "text",
    "stars",
    "business_id"
]

missing = [
    col
    for col in required_columns
    if col not in df_original.columns
]

if missing:

    raise RuntimeError(
        f"Faltan columnas necesarias: {missing}"
    )


# ================================================================
# 6. COPIA DE TRABAJO
# ================================================================

df = df_original.copy(
    deep=True
)


# ================================================================
# 7. ELIMINAR SOLO DUPLICADOS EXACTOS DE TEXTO
# ================================================================

before_dedup = len(df)

df = (
    df
    .drop_duplicates(
        subset=["text"],
        keep="first"
    )
    .reset_index(drop=True)
)

removed_duplicates = (
    before_dedup
    - len(df)
)

print(
    "\nDuplicados exactos de texto eliminados:",
    removed_duplicates
)

print(
    "Filas de trabajo:",
    len(df)
)


# ================================================================
# 8. CREAR ETIQUETA PROXY
# ================================================================

def stars_to_sentiment_id(stars):

    stars = int(stars)

    if stars <= 2:
        return 0

    elif stars == 3:
        return 1

    else:
        return 2


df["sentiment_id"] = (
    df["stars"]
    .apply(
        stars_to_sentiment_id
    )
)

ID2LABEL = {
    0: "NEGATIVO",
    1: "NEUTRAL",
    2: "POSITIVO"
}

df["sentiment"] = (
    df["sentiment_id"]
    .map(ID2LABEL)
)


print("\nDistribución proxy:")

print(
    df["sentiment"]
    .value_counts()
)


# ================================================================
# 9. LIMPIEZA BERT
# ================================================================

def clean_text_bert(text):

    text = str(text)

    text = html.unescape(
        text
    )

    text = re.sub(
        r"[\x00-\x1F\x7F]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


df["text_bert"] = (
    df["text"]
    .astype(str)
    .apply(
        clean_text_bert
    )
)


empty_texts = (
    df["text_bert"]
    .str.strip()
    .eq("")
    .sum()
)

if empty_texts > 0:

    df = (
        df[
            ~df["text_bert"]
            .str.strip()
            .eq("")
        ]
        .reset_index(drop=True)
    )


print(
    "Textos vacíos:",
    empty_texts
)


# ================================================================
# 10. CREAR SPLIT PARA MODELO DE DESPLIEGUE
# ================================================================
#
# IMPORTANTE:
# ESTE SPLIT NO ES EL TEST ACADÉMICO.
#
# Es exclusivamente un holdout interno para reconstruir
# el checkpoint de aplicación.
#
# El resultado NO sustituye las métricas oficiales G8.
# ================================================================

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

train_indices, validation_indices = next(
    sgkf.split(
        X=df["text_bert"],
        y=df["sentiment_id"],
        groups=df["business_id"]
    )
)

train_df = (
    df.iloc[
        train_indices
    ]
    .reset_index(drop=True)
)

validation_df = (
    df.iloc[
        validation_indices
    ]
    .reset_index(drop=True)
)


print("\n====================================================")
print("4. SPLIT DE DESPLIEGUE")
print("====================================================")

print(
    "TRAIN:",
    len(train_df)
)

print(
    "VALIDATION:",
    len(validation_df)
)

business_overlap = set(
    train_df["business_id"]
).intersection(
    set(
        validation_df["business_id"]
    )
)

print(
    "business_id compartidos:",
    len(business_overlap)
)


# ================================================================
# 11. GUARDAR SPLITS EN DRIVE
# ================================================================

train_path = (
    DATA_DIR
    / "deployment_train.csv"
)

validation_path = (
    DATA_DIR
    / "deployment_validation.csv"
)

train_df.to_csv(
    train_path,
    index=False
)

validation_df.to_csv(
    validation_path,
    index=False
)


# ================================================================
# 12. CONFIGURACIÓN BERT
# ================================================================

MODEL_NAME = (
    "google-bert/bert-base-uncased"
)

MAX_LENGTH = 256

BATCH_SIZE = 8

GRAD_ACCUM = 2

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.1

EPOCHS = 3

GRAD_CLIP = 1.0


# ================================================================
# 13. COMPROBAR SI YA EXISTE MODELO EN DRIVE
# ================================================================

model_exists = (
    (MODEL_DIR / "config.json").exists()
    and (
        (MODEL_DIR / "model.safetensors").exists()
        or
        (MODEL_DIR / "pytorch_model.bin").exists()
    )
)

tokenizer_exists = (
    (TOKENIZER_DIR / "tokenizer_config.json").exists()
)


# ================================================================
# 14. DISPOSITIVO
# ================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\n====================================================")
print("5. ESTADO DEL MODELO")
print("====================================================")

print(
    "Dispositivo:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "Modelo permanente existe:",
    model_exists
)

print(
    "Tokenizer permanente existe:",
    tokenizer_exists
)


# ================================================================
# 15. SI YA EXISTE, CARGARLO
# ================================================================

if model_exists and tokenizer_exists:

    print(
        "\nModelo encontrado en Drive."
    )

    print(
        "No se entrenará nuevamente."
    )

    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_DIR
    )

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_DIR
        )
        .to(device)
    )

    model.eval()


# ================================================================
# 16. SI NO EXISTE Y NO HAY GPU, DETENER AQUÍ
# ================================================================

elif not torch.cuda.is_available():

    print("\n====================================================")
    print("MODELO TODAVÍA NO EXISTE")
    print("====================================================")

    print(
        "Todo quedó preparado y guardado en Google Drive."
    )

    print(
        "\nActualmente Colab no te está dando GPU."
    )

    print(
        "NO voy a entrenar BERT en CPU porque sería muy lento."
    )

    print(
        "\nCuando Colab vuelva a darte una T4, "
        "abre ESTE MISMO cuaderno y ejecuta "
        "ESTA MISMA CELDA."
    )

    print(
        "\nEl código detectará automáticamente "
        "que los datos ya están en Drive y continuará "
        "desde el entrenamiento."
    )

    print(
        "\nNO tienes que volver a copiar nada."
    )

    raise SystemExit(
        "Preparación completada. "
        "Esperando GPU para entrenamiento."
    )


# ================================================================
# 17. ENTRENAR MODELO SI NO EXISTE Y HAY GPU
# ================================================================

else:

    print("\n====================================================")
    print("6. ENTRENANDO MODELO DE DESPLIEGUE")
    print("====================================================")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )


    class YelpDataset(Dataset):

        def __init__(
            self,
            texts,
            labels
        ):

            self.texts = list(
                texts
            )

            self.labels = np.asarray(
                labels,
                dtype=np.int64
            )


        def __len__(
            self
        ):

            return len(
                self.labels
            )


        def __getitem__(
            self,
            idx
        ):

            enc = tokenizer(
                self.texts[idx],
                truncation=True,
                max_length=MAX_LENGTH,
                padding=False
            )

            enc["labels"] = int(
                self.labels[idx]
            )

            return enc


    train_dataset = YelpDataset(
        train_df["text_bert"],
        train_df["sentiment_id"]
    )

    validation_dataset = YelpDataset(
        validation_df["text_bert"],
        validation_df["sentiment_id"]
    )


    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        return_tensors="pt"
    )


    generator = torch.Generator()
    generator.manual_seed(SEED)


    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        generator=generator
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator
    )


    # ------------------------------------------------------------
    # PESOS DE CLASE
    # ------------------------------------------------------------

    class_weights_np = compute_class_weight(
        class_weight="balanced",
        classes=np.array(
            [0, 1, 2]
        ),
        y=train_df[
            "sentiment_id"
        ].to_numpy()
    )

    class_weights = torch.tensor(
        class_weights_np,
        dtype=torch.float32,
        device=device
    )


    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,
            num_labels=3,
            id2label=ID2LABEL,
            label2id={
                v: k
                for k, v
                in ID2LABEL.items()
            }
        )
        .to(device)
    )


    criterion = torch.nn.CrossEntropyLoss(
        weight=class_weights
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )


    updates_per_epoch = math.ceil(
        len(train_loader)
        / GRAD_ACCUM
    )

    total_steps = (
        updates_per_epoch
        * EPOCHS
    )

    warmup_steps = int(
        total_steps
        * WARMUP_RATIO
    )


    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
    )


    scaler = torch.amp.GradScaler(
        "cuda"
    )


    best_f1 = -1.0
    best_epoch = None

    TEMP_CHECKPOINT = (
        PROJECT_DIR
        / "bert_temp_best.pt"
    )


    # ------------------------------------------------------------
    # ENTRENAMIENTO
    # ------------------------------------------------------------

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        model.train()

        optimizer.zero_grad()

        train_loss_sum = 0.0
        train_n = 0


        for step, batch in enumerate(
            train_loader,
            start=1
        ):

            labels = (
                batch.pop("labels")
                .to(device)
            )

            batch = {
                k: v.to(device)
                for k, v
                in batch.items()
            }


            with torch.amp.autocast(
                "cuda"
            ):

                logits = model(
                    **batch
                ).logits

                loss = criterion(
                    logits,
                    labels
                )

                backward_loss = (
                    loss
                    / GRAD_ACCUM
                )


            scaler.scale(
                backward_loss
            ).backward()


            train_loss_sum += (
                loss.item()
                * labels.size(0)
            )

            train_n += (
                labels.size(0)
            )


            update_now = (
                step % GRAD_ACCUM == 0
                or
                step == len(train_loader)
            )


            if update_now:

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                scheduler.step()

                optimizer.zero_grad()


        # --------------------------------------------------------
        # VALIDATION
        # --------------------------------------------------------

        model.eval()

        val_true = []
        val_pred = []


        with torch.no_grad():

            for batch in validation_loader:

                labels = (
                    batch.pop("labels")
                    .to(device)
                )

                batch = {
                    k: v.to(device)
                    for k, v
                    in batch.items()
                }


                with torch.amp.autocast(
                    "cuda"
                ):

                    logits = model(
                        **batch
                    ).logits


                preds = torch.argmax(
                    logits,
                    dim=1
                )


                val_true.extend(
                    labels.cpu()
                    .numpy()
                    .tolist()
                )

                val_pred.extend(
                    preds.cpu()
                    .numpy()
                    .tolist()
                )


        val_f1 = f1_score(
            val_true,
            val_pred,
            average="macro",
            labels=[0, 1, 2],
            zero_division=0
        )


        print(
            f"Época {epoch} | "
            f"train_loss="
            f"{train_loss_sum/train_n:.4f} | "
            f"val_f1_macro="
            f"{val_f1:.4f}"
        )


        if val_f1 > best_f1:

            best_f1 = val_f1

            best_epoch = epoch

            torch.save(
                model.state_dict(),
                TEMP_CHECKPOINT
            )


    # ------------------------------------------------------------
    # RECUPERAR MEJOR CHECKPOINT
    # ------------------------------------------------------------

    model.load_state_dict(
        torch.load(
            TEMP_CHECKPOINT,
            map_location=device
        )
    )

    model.eval()


    # ------------------------------------------------------------
    # GUARDAR PERMANENTEMENTE EN DRIVE
    # ------------------------------------------------------------

    model.save_pretrained(
        MODEL_DIR
    )

    tokenizer.save_pretrained(
        TOKENIZER_DIR
    )


    deployment_config = {

        "purpose":
            "deployment_model",

        "academic_results_replaced":
            False,

        "source_model":
            MODEL_NAME,

        "seed":
            SEED,

        "max_length":
            MAX_LENGTH,

        "batch_size":
            BATCH_SIZE,

        "gradient_accumulation":
            GRAD_ACCUM,

        "learning_rate":
            LEARNING_RATE,

        "epochs":
            EPOCHS,

        "best_epoch":
            best_epoch,

        "validation_f1_macro_deployment":
            float(best_f1)
    }


    with open(
        RESULTS_DIR
        / "deployment_config.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            deployment_config,
            f,
            indent=2,
            ensure_ascii=False
        )


    if TEMP_CHECKPOINT.exists():
        TEMP_CHECKPOINT.unlink()


    print("\nMODELO GUARDADO PERMANENTEMENTE:")
    print(MODEL_DIR)

    print("\nTOKENIZER GUARDADO:")
    print(TOKENIZER_DIR)


# ================================================================
# 18. FUNCIÓN DE INFERENCIA MASIVA
# ================================================================

def predict_batch(
    texts,
    batch_size=32
):

    model.eval()

    texts = [
        clean_text_bert(x)
        for x in texts
    ]

    all_pred = []
    all_probs = []


    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = (
            texts[
                start:
                start + batch_size
            ]
        )


        enc = tokenizer(
            batch_texts,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt"
        )


        enc = {
            k: v.to(device)
            for k, v
            in enc.items()
        }


        with torch.no_grad():

            logits = model(
                **enc
            ).logits

            probs = torch.softmax(
                logits,
                dim=1
            )

            preds = torch.argmax(
                probs,
                dim=1
            )


        all_pred.extend(
            preds.cpu()
            .numpy()
            .tolist()
        )

        all_probs.extend(
            probs.cpu()
            .numpy()
            .tolist()
        )


    return (
        np.array(
            all_pred
        ),
        np.array(
            all_probs
        )
    )


# ================================================================
# 19. INDIVIDUAL
# ================================================================

def analyze_single(
    text
):

    if text is None or not str(
        text
    ).strip():

        return (
            "Introduce una reseña.",
            {},
            ""
        )


    pred, probs = predict_batch(
        [text],
        batch_size=1
    )


    pred_id = int(
        pred[0]
    )

    p = probs[0]


    return (
        ID2LABEL[
            pred_id
        ],

        {
            "NEGATIVO":
                float(p[0]),

            "NEUTRAL":
                float(p[1]),

            "POSITIVO":
                float(p[2])
        },

        (
            f"{float(np.max(p))*100:.2f}%"
        )
    )


# ================================================================
# 20. CARGA DE ARCHIVOS
# ================================================================

def load_file(
    filepath
):

    filepath = str(
        filepath
    )


    if filepath.lower().endswith(
        ".csv"
    ):

        try:

            return pd.read_csv(
                filepath
            )

        except UnicodeDecodeError:

            return pd.read_csv(
                filepath,
                encoding="latin-1"
            )


    elif filepath.lower().endswith(
        (".xlsx", ".xls")
    ):

        return pd.read_excel(
            filepath
        )


    raise ValueError(
        "Archivo no compatible."
    )


# ================================================================
# 21. ANÁLISIS EJECUTIVO
# ================================================================

def build_executive_analysis(
    summary,
    avg_confidence,
    low_conf_pct
):

    values = dict(
        zip(
            summary[
                "Sentimiento"
            ],
            summary[
                "Porcentaje"
            ]
        )
    )


    neg = values.get(
        "NEGATIVO",
        0.0
    )

    neu = values.get(
        "NEUTRAL",
        0.0
    )

    pos = values.get(
        "POSITIVO",
        0.0
    )


    dominant = max(
        values,
        key=values.get
    )


    return f"""
RESUMEN EJECUTIVO

El sentimiento predominante es {dominant}.

Distribución de las predicciones:

• POSITIVO: {pos:.2f} %
• NEUTRAL: {neu:.2f} %
• NEGATIVO: {neg:.2f} %

La confianza promedio del modelo es
{avg_confidence*100:.2f} %.

El {low_conf_pct:.2f} % de las reseñas presenta
una confianza inferior al 60 % y debería considerarse
para revisión manual.

LECTURA PARA DECISIÓN

Las reseñas NEGATIVAS ({neg:.2f} %) constituyen el grupo
prioritario para identificar experiencias desfavorables.

Las reseñas NEUTRALES ({neu:.2f} %) deben interpretarse
con cautela porque esta fue la clase más difícil durante
la evaluación académica.

Una proporción elevada de opiniones POSITIVAS no elimina
la necesidad de estudiar el contenido de las reseñas
NEGATIVAS y los casos de baja confianza.

Estos porcentajes representan predicciones del modelo
sobre el archivo cargado y no constituyen por sí solos
una explicación causal del comportamiento del cliente.
""".strip()


# ================================================================
# 22. ANÁLISIS MASIVO
# ================================================================

def analyze_dataset(
    filepath
):

    if filepath is None:

        raise gr.Error(
            "Carga un archivo."
        )


    data = load_file(
        filepath
    )


    # Detección automática de columna de texto
    if "text" in data.columns:

        text_col = "text"

    else:

        object_cols = (
            data
            .select_dtypes(
                include="object"
            )
            .columns
            .tolist()
        )

        if not object_cols:

            raise gr.Error(
                "No encuentro una columna de texto."
            )

        text_col = object_cols[0]


    work = data.copy()


    valid = (
        work[text_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )


    analyzed = (
        work.loc[
            valid
        ]
        .copy()
    )


    texts = (
        analyzed[
            text_col
        ]
        .astype(str)
        .tolist()
    )


    pred, probs = predict_batch(
        texts
    )


    analyzed[
        "sentiment_predicted"
    ] = [
        ID2LABEL[
            int(x)
        ]
        for x in pred
    ]


    analyzed[
        "confidence"
    ] = probs.max(
        axis=1
    )


    analyzed[
        "prob_NEGATIVO"
    ] = probs[:, 0]

    analyzed[
        "prob_NEUTRAL"
    ] = probs[:, 1]

    analyzed[
        "prob_POSITIVO"
    ] = probs[:, 2]


    counts = (
        analyzed[
            "sentiment_predicted"
        ]
        .value_counts()
        .reindex(
            [
                "NEGATIVO",
                "NEUTRAL",
                "POSITIVO"
            ],
            fill_value=0
        )
    )


    percentages = (
        counts
        / len(analyzed)
        * 100
    )


    summary = pd.DataFrame({

        "Sentimiento":
            counts.index,

        "Cantidad":
            counts.values,

        "Porcentaje":
            percentages
            .round(2)
            .values
    })


    avg_conf = analyzed[
        "confidence"
    ].mean()


    low_conf = (
        analyzed[
            "confidence"
        ] < 0.60
    ).mean() * 100


    executive = build_executive_analysis(
        summary,
        avg_conf,
        low_conf
    )


    negative = (
        analyzed[
            analyzed[
                "sentiment_predicted"
            ] == "NEGATIVO"
        ]
        .sort_values(
            "confidence",
            ascending=False
        )
        .head(25)
    )


    output_file = (
        RESULTS_DIR
        / "analisis_masivo.xlsx"
    )


    with pd.ExcelWriter(
        output_file,
        engine="openpyxl"
    ) as writer:

        analyzed.to_excel(
            writer,
            sheet_name="Predicciones",
            index=False
        )

        summary.to_excel(
            writer,
            sheet_name="Resumen",
            index=False
        )

        negative.to_excel(
            writer,
            sheet_name="Negativos_prioritarios",
            index=False
        )

        pd.DataFrame({
            "Analisis":
                executive.split(
                    "\n"
                )
        }).to_excel(
            writer,
            sheet_name="Analisis_ejecutivo",
            index=False
        )


    return (
        summary,
        executive,
        negative.head(20),
        str(output_file)
    )


# ================================================================
# 23. GRADIO
# ================================================================

with gr.Blocks(
    title="Analítica de Sentimientos Yelp"
) as demo:

    gr.Markdown(
        """
# Analítica de sentimientos en reseñas de Yelp

Modelo BERT para análisis individual y masivo.

**BERT utiliza exclusivamente el texto como predictor.**
"""
    )


    with gr.Tab(
        "Análisis individual"
    ):

        input_text = gr.Textbox(
            label="Reseña",
            lines=6
        )

        button_single = gr.Button(
            "Analizar",
            variant="primary"
        )

        output_sentiment = gr.Textbox(
            label="Sentimiento"
        )

        output_probs = gr.Label(
            label="Probabilidades",
            num_top_classes=3
        )

        output_confidence = gr.Textbox(
            label="Confianza"
        )


        button_single.click(
            analyze_single,
            inputs=input_text,
            outputs=[
                output_sentiment,
                output_probs,
                output_confidence
            ]
        )


    with gr.Tab(
        "Análisis masivo"
    ):

        input_file = gr.File(
            label="Subir CSV o Excel",
            type="filepath"
        )

        button_dataset = gr.Button(
            "Analizar dataset",
            variant="primary"
        )

        output_summary = gr.Dataframe(
            label="Distribución de sentimientos"
        )

        output_executive = gr.Textbox(
            label="Análisis ejecutivo",
            lines=20
        )

        output_negative = gr.Dataframe(
            label="Reseñas negativas prioritarias"
        )

        output_download = gr.File(
            label="Descargar análisis completo"
        )


        button_dataset.click(
            analyze_dataset,
            inputs=input_file,
            outputs=[
                output_summary,
                output_executive,
                output_negative,
                output_download
            ]
        )


# ================================================================
# 24. ESTADO FINAL
# ================================================================

print("\n====================================================")
print("APLICACIÓN LISTA")
print("====================================================")

print(
    "Modelo permanente:",
    MODEL_DIR
)

print(
    "Tokenizer permanente:",
    TOKENIZER_DIR
)

print(
    "Dataset permanente:",
    dataset_path
)

print(
    "Aplicación individual:",
    "SÍ"
)

print(
    "Aplicación masiva:",
    "SÍ"
)

print(
    "Modelo académico reemplazado:",
    "NO"
)


demo.launch(
    share=True,
    debug=True
)

0. PREPARANDO ENTORNO

1. GOOGLE DRIVE
Mounted at /content/drive
Proyecto: /content/drive/MyDrive/Yelp_NLP_Final
Datos: /content/drive/MyDrive/Yelp_NLP_Final/data
Modelo: /content/drive/MyDrive/Yelp_NLP_Final/bert_model
Tokenizer: /content/drive/MyDrive/Yelp_NLP_Final/tokenizer
Resultados: /content/drive/MyDrive/Yelp_NLP_Final/results

2. BUSCANDO DATASET
Dataset localizado: /content/drive/MyDrive/Yelp_NLP_Final/data/yelp_dataset.xlsx
Dataset permanente: /content/drive/MyDrive/Yelp_NLP_Final/data/yelp_dataset.xlsx

3. CARGANDO DATASET
Filas originales: 10000
Columnas: ['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id', 'cool', 'useful', 'funny']

Duplicados exactos de texto eliminados: 2
Filas de trabajo: 9998

Distribución proxy:
sentiment
POSITIVO    6862
NEGATIVO    1675
NEUTRAL     1461
Name: count, dtype: int64
Textos vacíos: 0

4. SPLIT DE DESPLIEGUE
TRAIN: 7899
VALIDATION: 2099
business_id compartidos: 0

5. ESTADO DEL MODELO
Dispositivo: cuda
GPU: Tesla T4


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]


APLICACIÓN LISTA
Modelo permanente: /content/drive/MyDrive/Yelp_NLP_Final/bert_model
Tokenizer permanente: /content/drive/MyDrive/Yelp_NLP_Final/tokenizer
Dataset permanente: /content/drive/MyDrive/Yelp_NLP_Final/data/yelp_dataset.xlsx
Aplicación individual: SÍ
Aplicación masiva: SÍ
Modelo académico reemplazado: NO
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://80f398642d9ec7c5bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://80f398642d9ec7c5bf.gradio.live


In [1]:
import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: NO DISPONIBLE")

CUDA disponible: True
GPU: Tesla T4


In [4]:
import pandas as pd
from pathlib import Path

archivo = Path("/content/drive/MyDrive/Yelp_NLP_Final/data/yelp_dataset.xlsx")

print("=== COMPROBACIÓN RÁPIDA ===")
print("Archivo existe:", archivo.exists())
print("Tamaño MB:", round(archivo.stat().st_size / 1024**2, 2))

df_check = pd.read_excel(
    archivo,
    usecols=["text"]
)

print("Filas:", len(df_check))
print(
    "Textos no vacíos:",
    df_check["text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

print("=== FIN ===")

=== COMPROBACIÓN RÁPIDA ===
Archivo existe: True
Tamaño MB: 3.77


Exception ignored in: <function ZipFile.__del__ at 0x7d2b0790a200>
Traceback (most recent call last):
  File "/usr/lib/python3.13/zipfile/__init__.py", line 2015, in __del__
    def __del__(self):
KeyboardInterrupt: 


Filas: 10000
Textos no vacíos: 10000
=== FIN ===


In [5]:
import pandas as pd
from pathlib import Path

archivo = Path(
    "/content/drive/MyDrive/Yelp_NLP_Final/results/analisis_masivo.xlsx"
)

print("=== RESULTADO DEL ANÁLISIS MASIVO ===")
print("Archivo existe:", archivo.exists())

if archivo.exists():
    resumen = pd.read_excel(
        archivo,
        sheet_name="Resumen"
    )

    print("\nRESUMEN:")
    print(resumen.to_string(index=False))

    print("\nTOTAL ANALIZADO:", int(resumen["Cantidad"].sum()))

=== RESULTADO DEL ANÁLISIS MASIVO ===
Archivo existe: True

RESUMEN:
Sentimiento  Cantidad  Porcentaje
   NEGATIVO      1708       17.08
    NEUTRAL      1848       18.48
   POSITIVO      6444       64.44

TOTAL ANALIZADO: 10000


In [6]:
# ============================================================
# ANÁLISIS DE CAUSAS — RESEÑAS NEGATIVAS
# ============================================================

import pandas as pd
import re
from pathlib import Path

ARCHIVO = Path(
    "/content/drive/MyDrive/Yelp_NLP_Final/results/analisis_masivo.xlsx"
)

print("=== ANÁLISIS DE RESEÑAS NEGATIVAS ===")

# Recuperar predicciones ya generadas
df = pd.read_excel(
    ARCHIVO,
    sheet_name="Predicciones"
)

negativas = df[
    df["sentiment_predicted"] == "NEGATIVO"
].copy()

print("Total analizado:", len(df))
print("Reseñas negativas:", len(negativas))

# ------------------------------------------------------------
# Categorías operativas
# ------------------------------------------------------------

categorias = {

    "Servicio / atención": [
        "service", "staff", "waiter", "waitress",
        "server", "manager", "employee",
        "rude", "customer service"
    ],

    "Tiempo / demora": [
        "wait", "waiting", "slow", "minutes",
        "hour", "late", "forever"
    ],

    "Comida / producto": [
        "food", "meal", "dish", "taste",
        "cold", "dry", "bad", "bland",
        "overcooked", "undercooked"
    ],

    "Precio / valor": [
        "price", "expensive", "overpriced",
        "cost", "money", "worth"
    ],

    "Pedido / entrega": [
        "order", "delivery", "delivered",
        "wrong order", "takeout", "take out"
    ],

    "Limpieza / instalaciones": [
        "dirty", "clean", "bathroom",
        "restroom", "table", "smell",
        "parking"
    ]
}


def detectar_categoria(texto, palabras):

    texto = str(texto).lower()

    return any(
        re.search(
            r"\b" + re.escape(p) + r"\b",
            texto
        )
        for p in palabras
    )


for categoria, palabras in categorias.items():

    negativas[categoria] = negativas["text"].apply(
        lambda x: detectar_categoria(x, palabras)
    )


# ------------------------------------------------------------
# Resumen
# ------------------------------------------------------------

resultados = []

for categoria in categorias:

    cantidad = int(
        negativas[categoria].sum()
    )

    porcentaje = (
        cantidad / len(negativas) * 100
        if len(negativas)
        else 0
    )

    resultados.append({
        "Categoría": categoria,
        "Reseñas": cantidad,
        "% de negativas": round(porcentaje, 2)
    })


resumen_causas = (
    pd.DataFrame(resultados)
    .sort_values(
        "Reseñas",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n=== PRINCIPALES TEMAS EN RESEÑAS NEGATIVAS ===")
print(
    resumen_causas.to_string(index=False)
)


# ------------------------------------------------------------
# Guardar resultado
# ------------------------------------------------------------

SALIDA = Path(
    "/content/drive/MyDrive/Yelp_NLP_Final/results/"
    "analisis_causas_negativas.xlsx"
)

with pd.ExcelWriter(
    SALIDA,
    engine="openpyxl"
) as writer:

    resumen_causas.to_excel(
        writer,
        sheet_name="Resumen_causas",
        index=False
    )

    negativas.to_excel(
        writer,
        sheet_name="Negativas_clasificadas",
        index=False
    )


print("\nArchivo guardado:")
print(SALIDA)

print("\n=== ANÁLISIS TERMINADO ===")

=== ANÁLISIS DE RESEÑAS NEGATIVAS ===
Total analizado: 10000
Reseñas negativas: 1708

=== PRINCIPALES TEMAS EN RESEÑAS NEGATIVAS ===
               Categoría  Reseñas  % de negativas
       Comida / producto     1008           59.02
     Servicio / atención      834           48.83
         Tiempo / demora      537           31.44
          Precio / valor      478           27.99
Limpieza / instalaciones      377           22.07
        Pedido / entrega      295           17.27

Archivo guardado:
/content/drive/MyDrive/Yelp_NLP_Final/results/analisis_causas_negativas.xlsx

=== ANÁLISIS TERMINADO ===


In [ ]:
# ============================================================
# APLICACIÓN FINAL — ANALÍTICA EJECUTIVA DE SENTIMIENTOS
# No entrena ni modifica BERT
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path
import gradio as gr

RESULTS_DIR = Path(
    "/content/drive/MyDrive/Yelp_NLP_Final/results"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# CATEGORÍAS TEMÁTICAS
# ------------------------------------------------------------

CATEGORIAS = {
    "Comida / producto": [
        "food", "meal", "dish", "taste", "cold",
        "dry", "bad", "bland", "overcooked", "undercooked"
    ],
    "Servicio / atención": [
        "service", "staff", "waiter", "waitress",
        "server", "manager", "employee", "rude",
        "customer service"
    ],
    "Tiempo / demora": [
        "wait", "waiting", "slow", "minutes",
        "hour", "late", "forever"
    ],
    "Precio / valor": [
        "price", "expensive", "overpriced",
        "cost", "money", "worth"
    ],
    "Limpieza / instalaciones": [
        "dirty", "clean", "bathroom", "restroom",
        "table", "smell", "parking"
    ],
    "Pedido / entrega": [
        "order", "delivery", "delivered",
        "wrong order", "takeout", "take out"
    ]
}


def detectar_tema(texto, palabras):
    texto = str(texto).lower()

    return any(
        re.search(r"\b" + re.escape(p) + r"\b", texto)
        for p in palabras
    )


def cargar_archivo(filepath):
    filepath = str(filepath)

    if filepath.lower().endswith(".csv"):
        try:
            return pd.read_csv(filepath)
        except UnicodeDecodeError:
            return pd.read_csv(filepath, encoding="latin-1")

    if filepath.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(filepath)

    raise ValueError("Formato no compatible. Usa CSV o Excel.")


# ------------------------------------------------------------
# ANÁLISIS MASIVO COMPLETO
# ------------------------------------------------------------

def analisis_ejecutivo_masivo(filepath):

    if filepath is None:
        raise gr.Error("Debes cargar un archivo CSV o Excel.")

    df = cargar_archivo(filepath)

    # Detectar columna de texto
    if "text" in df.columns:
        text_col = "text"
    else:
        posibles = df.select_dtypes(include="object").columns.tolist()

        if not posibles:
            raise gr.Error("No se encontró una columna de texto.")

        text_col = posibles[0]

    work = df.copy()

    validos = (
        work[text_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    work = work.loc[validos].copy()

    if len(work) == 0:
        raise gr.Error("No existen textos válidos para analizar.")

    # --------------------------------------------------------
    # BERT — INFERENCIA
    # --------------------------------------------------------

    pred, probs = predict_batch(
        work[text_col].astype(str).tolist()
    )

    etiquetas = {
        0: "NEGATIVO",
        1: "NEUTRAL",
        2: "POSITIVO"
    }

    work["sentiment_predicted"] = [
        etiquetas[int(x)] for x in pred
    ]

    work["confidence"] = probs.max(axis=1)

    work["prob_NEGATIVO"] = probs[:, 0]
    work["prob_NEUTRAL"] = probs[:, 1]
    work["prob_POSITIVO"] = probs[:, 2]

    # --------------------------------------------------------
    # DISTRIBUCIÓN GLOBAL
    # --------------------------------------------------------

    counts = (
        work["sentiment_predicted"]
        .value_counts()
        .reindex(
            ["NEGATIVO", "NEUTRAL", "POSITIVO"],
            fill_value=0
        )
    )

    resumen = pd.DataFrame({
        "Sentimiento": counts.index,
        "Cantidad": counts.values,
        "Porcentaje": (
            counts.values / len(work) * 100
        ).round(2)
    })

    # --------------------------------------------------------
    # RESEÑAS NEGATIVAS
    # --------------------------------------------------------

    negativas = work[
        work["sentiment_predicted"] == "NEGATIVO"
    ].copy()

    causas = []

    for categoria, palabras in CATEGORIAS.items():

        negativas[categoria] = negativas[text_col].apply(
            lambda x: detectar_tema(x, palabras)
        )

        cantidad = int(negativas[categoria].sum())

        porcentaje = (
            cantidad / len(negativas) * 100
            if len(negativas) else 0
        )

        causas.append({
            "Tema detectado": categoria,
            "Reseñas negativas": cantidad,
            "% de negativas": round(porcentaje, 2)
        })

    resumen_causas = (
        pd.DataFrame(causas)
        .sort_values("Reseñas negativas", ascending=False)
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # INDICADORES
    # --------------------------------------------------------

    total = len(work)

    positivo = int(counts["POSITIVO"])
    neutral = int(counts["NEUTRAL"])
    negativo = int(counts["NEGATIVO"])

    pct_pos = positivo / total * 100
    pct_neu = neutral / total * 100
    pct_neg = negativo / total * 100

    confianza_promedio = work["confidence"].mean() * 100

    baja_confianza = (
        (work["confidence"] < 0.60).mean() * 100
    )

    if len(resumen_causas):
        principal = resumen_causas.iloc[0]
        tema_principal = principal["Tema detectado"]
        pct_tema = principal["% de negativas"]
    else:
        tema_principal = "Sin información"
        pct_tema = 0

    # --------------------------------------------------------
    # TEXTO EJECUTIVO
    # --------------------------------------------------------

    informe = f"""
ANÁLISIS EJECUTIVO

Se analizaron {total:,} reseñas mediante el modelo BERT de despliegue.

DISTRIBUCIÓN DE SENTIMIENTOS

• POSITIVO: {positivo:,} reseñas ({pct_pos:.2f}%)
• NEUTRAL: {neutral:,} reseñas ({pct_neu:.2f}%)
• NEGATIVO: {negativo:,} reseñas ({pct_neg:.2f}%)

LECTURA GENERAL

El sentimiento predominante es POSITIVO, con {pct_pos:.2f}% de las
reseñas analizadas. No obstante, {negativo:,} reseñas ({pct_neg:.2f}%)
fueron clasificadas como NEGATIVAS y constituyen el grupo prioritario
para análisis de experiencia del cliente.

PRINCIPALES TEMAS EN LAS RESEÑAS NEGATIVAS

El tema detectado con mayor frecuencia es "{tema_principal}",
presente en {pct_tema:.2f}% de las reseñas negativas.

Los porcentajes temáticos no suman necesariamente 100%, porque una
misma reseña puede mencionar varios temas.

PRIORIZACIÓN

1. Revisar las reseñas negativas relacionadas con comida/producto.
2. Examinar problemas recurrentes de servicio y atención.
3. Identificar patrones relacionados con tiempos de espera.
4. Revisar percepción de precio y valor.
5. Dar seguimiento a limpieza, instalaciones y pedidos/entregas.

CALIDAD DE LA PREDICCIÓN

Confianza promedio: {confianza_promedio:.2f}%
Reseñas con confianza inferior al 60%: {baja_confianza:.2f}%

Los casos de baja confianza deberían revisarse manualmente.

NOTA METODOLÓGICA

Los temas se detectan mediante reglas de palabras clave sobre las
reseñas clasificadas como negativas. Su presencia indica asociación
temática, no demuestra causalidad. La clase NEUTRAL debe interpretarse
con especial cautela debido a su mayor dificultad observada durante
la evaluación académica.
""".strip()

    # --------------------------------------------------------
    # RESEÑAS PRIORITARIAS
    # --------------------------------------------------------

    prioritarias = (
        negativas
        .sort_values("confidence", ascending=False)
        .head(30)
    )

    # --------------------------------------------------------
    # EXCEL FINAL
    # --------------------------------------------------------

    salida = RESULTS_DIR / "reporte_ejecutivo_sentimientos.xlsx"

    with pd.ExcelWriter(salida, engine="openpyxl") as writer:

        work.to_excel(
            writer,
            sheet_name="Predicciones",
            index=False
        )

        resumen.to_excel(
            writer,
            sheet_name="Sentimientos",
            index=False
        )

        resumen_causas.to_excel(
            writer,
            sheet_name="Temas_negativos",
            index=False
        )

        prioritarias.to_excel(
            writer,
            sheet_name="Negativas_prioritarias",
            index=False
        )

        pd.DataFrame({
            "Informe ejecutivo": informe.split("\n")
        }).to_excel(
            writer,
            sheet_name="Informe_ejecutivo",
            index=False
        )

    return (
        resumen,
        resumen_causas,
        informe,
        prioritarias,
        str(salida)
    )


# ------------------------------------------------------------
# INTERFAZ FINAL
# ------------------------------------------------------------

with gr.Blocks(
    title="Analítica Ejecutiva de Sentimientos Yelp"
) as app_final:

    gr.Markdown("""
# Analítica Ejecutiva de Sentimientos en Reseñas de Yelp

Carga un archivo CSV o Excel con reseñas.

El sistema utiliza **BERT exclusivamente sobre el texto** para
clasificar sentimientos y posteriormente realiza un análisis
agregado para apoyar la toma de decisiones.
""")

    archivo_input = gr.File(
        label="Subir dataset CSV o Excel",
        type="filepath"
    )

    analizar_btn = gr.Button(
        "Analizar dataset",
        variant="primary"
    )

    gr.Markdown("## Distribución de sentimientos")

    salida_sentimientos = gr.Dataframe()

    gr.Markdown("## Principales temas en reseñas negativas")

    salida_causas = gr.Dataframe()

    gr.Markdown("## Análisis ejecutivo")

    salida_informe = gr.Textbox(
        lines=22
    )

    gr.Markdown("## Reseñas negativas prioritarias")

    salida_prioritarias = gr.Dataframe()

    salida_excel = gr.File(
        label="Descargar reporte ejecutivo completo"
    )

    analizar_btn.click(
        fn=analisis_ejecutivo_masivo,
        inputs=archivo_input,
        outputs=[
            salida_sentimientos,
            salida_causas,
            salida_informe,
            salida_prioritarias,
            salida_excel
        ]
    )


print("==============================================")
print("APLICACIÓN EJECUTIVA PREPARADA")
print("==============================================")
print("BERT reentrenado: NO")
print("Modelo modificado: NO")
print("TEST académico utilizado: NO")
print("Análisis temático integrado: SÍ")
print("Reporte Excel integrado: SÍ")

app_final.launch(
    share=True,
    debug=True
)

APLICACIÓN EJECUTIVA PREPARADA
BERT reentrenado: NO
Modelo modificado: NO
TEST académico utilizado: NO
Análisis temático integrado: SÍ
Reporte Excel integrado: SÍ
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://caso-practico-yelp-sentimiento-popb3qpztrhzsggykpkio2.streamlit.app
